In [ ]:
import os
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from config import CRAWL_DOWNLOAD_DIR

# ====== 설정 (필요 시 config.py에서 값 수정) ======
DOWNLOAD_DIR = CRAWL_DOWNLOAD_DIR
LOGIN_URL = "https://adpanchok.co.kr/partner/login.php?"
CHECKPOINT_PATH = "progress_checkpoint.json"
DEFAULT_START_PAGE = 1
MAX_PAGE_SAFETY = 500  # 무한루프 방지용 안전 상한선 (실제 마지막 페이지는 자동 감지)

options = webdriver.ChromeOptions()
options.add_argument("--kiosk-printing")

prefs = {
    "printing.print_preview_sticky_settings.appState": """
    {
      "recentDestinations": [{
        "id": "Save as PDF",
        "origin": "local"
      }],
      "selectedDestinationId": "Save as PDF",
      "version": 2
    }
    """,
    "savefile.default_directory": DOWNLOAD_DIR
}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 20)

# 🔐 로그인 + 발주목록 이동 (수동 — 보안코드/OTP 때문에 자동화 불가)
driver.get(LOGIN_URL)
input("👉 로그인 후 발주목록 페이지까지 이동하고 엔터")

# 📍 체크포인트 로드/저장 (자동 재개)
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f).get("last_page", DEFAULT_START_PAGE)
    return DEFAULT_START_PAGE

def save_checkpoint(page_no):
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump({"last_page": page_no}, f)

START_PAGE = load_checkpoint()
print(f"▶ {START_PAGE} 페이지부터 시작 (체크포인트: {CHECKPOINT_PATH})")

# 📂 이미 받은 파일 기준 중복 스킵 목록
existing = set()
if os.path.isdir(DOWNLOAD_DIR):
    for f in os.listdir(DOWNLOAD_DIR):
        if f.startswith("고려기프트_") and f.endswith(".pdf"):
            no = f.replace("고려기프트_", "").replace(".pdf", "")
            if no.isdigit():
                existing.add(no)

attempt_count = 0

def go_page(page_no):
    driver.execute_script(f"getPageGo({page_no});")
    time.sleep(3)

page = START_PAGE
prev_order_nos = None

while page <= MAX_PAGE_SAFETY:
    # 페이지 처리 시작 시점에 체크포인트 저장
    # → 중간에 죽어도 다음 실행이 같은 페이지부터 다시 시도 (이미 받은 건 existing이 걸러줌)
    save_checkpoint(page)

    print(f"📄 {page} 페이지 처리 중")
    go_page(page)

    rows = driver.find_elements(By.TAG_NAME, "tr")

    # 이 페이지의 주문번호 목록 먼저 파악 (마지막 페이지 판단용)
    current_order_nos = set()
    for row in rows:
        tds = row.find_elements(By.TAG_NAME, "td")
        if len(tds) < 5:
            continue
        order_no = tds[0].text.strip()
        if order_no.isdigit():
            current_order_nos.add(order_no.zfill(5))

    if not current_order_nos or current_order_nos == prev_order_nos:
        print(f"🏁 {page} 페이지에서 더 이상 새 주문이 없어 마지막 페이지로 판단하고 종료합니다.")
        break

    prev_order_nos = current_order_nos

    for row in rows:
        try:
            tds = row.find_elements(By.TAG_NAME, "td")
            if len(tds) < 5:
                continue

            order_no = tds[0].text.strip()
            if not order_no.isdigit():
                continue

            order_no_fmt = order_no.zfill(5)

            if order_no_fmt in existing:
                print(f"⏭ 이미 있음: {order_no_fmt}")
                continue

            approve_btn = row.find_element(
                By.XPATH, ".//button[contains(@class,'status_ok') and text()='승인']"
            )

            approve_btn.click()
            attempt_count += 1

            print(f"📌 승인 클릭 {attempt_count}회 / 주문번호 {order_no}")
            time.sleep(3)

            # 발주서 창 열릴 때까지 대기
            main_window = driver.current_window_handle
            WebDriverWait(driver, 10).until(
                lambda d: len(d.window_handles) > 1
            )

            driver.switch_to.window(driver.window_handles[-1])
            time.sleep(1)

            before_files = set(os.listdir(DOWNLOAD_DIR))

            driver.execute_script("window.print();")

            pdf_path = None
            for _ in range(10):
                time.sleep(1)
                after_files = set(os.listdir(DOWNLOAD_DIR))
                new_files = after_files - before_files
                pdfs = [f for f in new_files if f.endswith(".pdf")]
                if pdfs:
                    pdf_path = pdfs[0]
                    break

            driver.close()
            driver.switch_to.window(main_window)

            # 📂 파일명 변경 (PDF 없으면 스킵)
            if pdf_path:
                new_name = f"고려기프트_{order_no_fmt}.pdf"
                os.rename(
                    os.path.join(DOWNLOAD_DIR, pdf_path),
                    os.path.join(DOWNLOAD_DIR, new_name)
                )
                existing.add(order_no_fmt)
                print(f"✅ 저장 완료: {new_name}")
            else:
                print("⚠ PDF 생성 실패 (저장 안 됨)")

            time.sleep(random.uniform(1.5, 2.5))

        except Exception as e:
            print("❌ 스킵:", e)

    page += 1

# 정상적으로 끝났으면 다음 실행을 위해 체크포인트를 마지막 지점으로 남겨둠
save_checkpoint(page)
print("🏁 완료. 체크포인트:", CHECKPOINT_PATH)